<a href="https://colab.research.google.com/github/jeffheaton/app_generative_ai/blob/main/t81_559_class_04_1_langchain_chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-559: Applications of Generative Artificial Intelligence
**Module 4: LangChain: Chat and Memory**
* Instructor: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.wustl.edu/Programs/Pages/default.aspx)
* For more information visit the [class website](https://sites.wustl.edu/jeffheaton/t81-558/).

# Module 4 Material

* **Part 4.1: LangChain Conversations** [[Video]](https://www.youtube.com/watch?v=-wXT2RlzJec&ab_channel=JeffHeaton) [[Notebook]](t81_559_class_04_1_langchain_chat.ipynb)
* Part 4.2: Conversation Buffer Window Memory [[Video]](https://www.youtube.com/watch?v=G-l3T1Z9CHc&ab_channel=JeffHeaton) [[Notebook]](t81_559_class_04_2_memory_buffer.ipynb)
* Part 4.3: Chat with Summary and Fixed Window [[Video]](https://www.youtube.com/watch?v=z0iTmoEgn9U&ab_channel=JeffHeaton) [[Notebook]](t81_559_class_04_3_summary.ipynb)
* Part 4.4: Chat with Persistence, Rollback and Regeneration [[Video]](https://www.youtube.com/watch?v=7QEFjNE6wxs&ab_channel=JeffHeaton) [[Notebook]](t81_559_class_04_4_persistence.ipynb)
* Part 4.5: Automated Coder Application [[Video]](https://www.youtube.com/watch?v=pHcKXOMDZKU&ab_channel=JeffHeaton) [[Notebook]](t81_559_class_04_5_coder.ipynb)

# Google CoLab Instructions

The following code ensures that Google CoLab is running and maps Google Drive if needed.

In [1]:
import os

try:
    from google.colab import drive, userdata
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# OpenAI Secrets
if COLAB:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Install needed libraries in CoLab
if COLAB:
    # !pip install langchain langchain_openai
    !pip install -U langchain-openai

Note: using Google CoLab
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 10.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1


# 4.1: LangChain Conversations

Large language models (LLMs) facilitate interaction like human conversations. They are capable of referencing information shared earlier in the dialogue. In this module, we will explore managing an LLM's memory capabilities. Notably, LLMs need an inherent memory system beyond their immediate context buffer. Consequently, it falls upon external systems like LangChain to embed all previous conversational memory into each new prompt. To effectively remember past interactions, LangChain maintains a comprehensive transcript that accumulates over time. This transcript is reintroduced to the LLM with each exchange, which incorporates both user inputs and LLM responses. With each new prompt, the LLM is tasked to generate a suitable next response, thereby perpetuating the interactive process.

## Creating a Chat Conversation

The code snippet provides two functions designed to create a basic conversation utility for a Large Language Model (LLM). This is part of an effort to build foundational utilities before introducing more complex memory capabilities using LangChain classes in later sections.

First, the necessary modules and classes are imported. HumanMessage and SystemMessage from langchain_core.messages are used to represent messages from a human user and system responses, respectively. The ChatPromptTemplate, HumanMessagePromptTemplate, and SystemMessagePromptTemplate from langchain_core.prompts.chat help format prompts for the chat. Additionally, ChatOpenAI from langchain_openai is used to interact with OpenAI's language models, and display_markdown from IPython.display allows for markdown rendering within IPython environments.

The first function, begin_conversation, initializes a conversation with a system prompt, which is a predefined message declaring that the system's role is to assist (specified by the DEFAULT_SYSTEM variable). It creates an initial SystemMessage containing this prompt and returns a list containing this message, setting the stage for a conversation.

The second function, converse, facilitates the interaction between the human user and the LLM. It takes an LLM instance, the ongoing conversation (a list of messages), and the user's prompt as inputs. It begins by appending the user's message (encapsulated as a HumanMessage) to the conversation. It then invokes the LLM to respond based on the entire conversation history up to that point. The LLM's response is captured and added back to the conversation. Finally, the content of the LLM's response is returned, providing the output for the user to see. This function enables a dynamic and continuous exchange between the user and the system, leveraging the LLM's capabilities to generate relevant responses.

In [2]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)
from langchain_openai import ChatOpenAI
from IPython.display import display_markdown

DEFAULT_SYSTEM = "You are a helpful assistant. Format answers with markdown."

def begin_conversation(sys_prompt):
  messages = [
      SystemMessage(content=sys_prompt)
  ]
  return messages

def converse(llm, conversation, prompt):
  conversation.append(HumanMessage(content=prompt))
  output = llm.invoke(conversation)
  conversation.append(output)
  return output.content

The provided code builds upon the initial snippet to create a simple conversation where the Large Language Model (LLM) recalls the user's name through a series of exchanges.

The conversation starts by initializing an instance of ChatOpenAI, a class from the LangChain library, with a specific model ('gpt-5-mini'). Several parameters are set for the LLM, including the temperature, which controls the randomness of the response, and n, which specifies the number of responses to generate (in this case, a single response).

The function begin_conversation is called with the constant DEFAULT_SYSTEM as its argument. This function initializes the conversation with a system message stating, "You are a helpful assistant." The conversation list, containing this initial system message, is created and will be used to track the entire conversation history.

Next, the converse function is used to facilitate the dialogue between the user and the LLM. The user begins by sending a prompt: "Hello, what is my name?" This user's message is appended to the conversation history, and the LLM is invoked to generate a response based on the conversation so far. The LLM's response is added to the conversation and displayed using display_markdown to format the output appropriately.

In response to realizing the LLM does not yet know the user's name, the user then provides their name with the statement, "Oh sorry, my name is Jeff." This message is similarly processed: added to the conversation, and the LLM generates a new response acknowledging the name or continuing the conversation. Again, the response is displayed.

Finally, the user asks again, "What is my name?" to test if the LLM recalls the name from earlier in the conversation. The same process ensues, with the LLM generating a response based on the updated conversation history that now includes the user's name.

This sequence effectively demonstrates a simple use case where the conversation history maintained in the list enables the LLM to recall and utilize context from earlier exchanges, such as remembering the user's name. The ability to recall details like this is crucial for creating more engaging and personalized user interactions with LLMs.

In [3]:
MODEL = 'gpt-5-mini'

# Initialize the OpenAI LLM with your API key
llm = ChatOpenAI(
  model=MODEL,
  temperature= 0.3,
  n= 1)

conversation = begin_conversation(DEFAULT_SYSTEM)
output = converse(llm, conversation, "Hello, what is my name?")
display_markdown(output,raw=True)
output = converse(llm, conversation, "Oh sorry, my name is Fiona.")
display_markdown(output,raw=True)
output = converse(llm, conversation, "What is my name?")
display_markdown(output,raw=True)

I don't know your name — I don't have access to personal data unless you tell me. What would you like me to call you? 

If you want, tell me your name or nickname and I’ll use it for this conversation. (I can’t remember it across separate sessions unless the app you’re using stores that info.)

Nice to meet you, Fiona — I’ll call you that in this conversation. How can I help you today?

Your name is Fiona. How can I help you today?

## Conversing with the LLM in Markdown

The function chat serves as a facilitator for conversation between a human user and a Large Language Model (LLM) and enhances the display of the chat responses using Markdown formatting. It is built upon the earlier provided code snippets that set up a basic conversational framework with an LLM.

Markdown is a lightweight markup language with plain-text formatting syntax. It is designed to be converted into HTML or other formats while remaining easy to read and write in its raw form. This feature makes it popular for writing on the web, as users can create formatted text (like headers, lists, italics, and bold text) using simple and readable symbols.

In the chat function, the process starts by printing the user's prompt prefixed with "Human: ". This mimics a real chat interface, clearly delineating the messages that are input by the user. Then, the function calls converse, which was previously described, to process the prompt within the ongoing conversation with the LLM. This function appends the human message to the conversation, invokes the LLM to generate a response based on the conversation's context, and then appends this system-generated message back into the conversation history.

After obtaining the LLM's response through converse, the function display_markdown is used to render this response. The raw=True parameter tells the IPython display function to treat the string as raw Markdown. By using Markdown formatting, responses can include enhanced textual features such as italics, bolding, and lists, which can make the output more readable and engaging.

By formatting LLM responses in Markdown, we reinforce the natural, text-based communication style of the LLM. Given that LLMs, such as those provided by OpenAI, often format their responses in Markdown to leverage its text-enhancement capabilities, this approach ensures that the conversation utility outputs responses that utilize the full range of expressive possibilities offered by Markdown, enhancing the user interaction experience. This design choice aligns well with the Markdown capabilities inherently supported by many LLMs, ensuring that the responses are both visually appealing and functionally informative.

In [4]:
def chat(llm, conversation, prompt):
  print(f"Human: {prompt}")
  output = converse(llm, conversation, prompt)
  display_markdown(output,raw=True)

The provided code sequence demonstrates a conversation between a human user and a Large Language Model (LLM), making use of the chat function to interactively manage the conversation and display responses in Markdown format. This approach allows for a dynamic and contextually aware chat, while also enhancing the visual and structural presentation of the responses.

In [5]:
conversation = begin_conversation(DEFAULT_SYSTEM)
chat(llm, conversation, "What is my name?")
chat(llm, conversation, "Okay, then let me introduce myself, my name is Fiona")
chat(llm, conversation, "What is my name?")
chat(llm, conversation, "Give me a table of the 5 most populus cities with population and country.")


Human: What is my name?


I don't know your name — I don't have access to personal data unless you tell me. If you'd like, you can tell me now (or give a nickname) and I'll use it for the rest of this conversation. I can't retain it across sessions, though.

If you were asking because you think the app might know it, check your account/profile or settings in the platform you're using.

Human: Okay, then let me introduce myself, my name is Fiona


Nice to meet you, Fiona! I'll call you that for this conversation. How can I help you today?

Human: What is my name?


Your name is Fiona.

Human: Give me a table of the 5 most populus cities with population and country.


Sure, Fiona — do you mean worldwide, and would you like city proper populations or metropolitan/urban-area populations? I’ll assume you mean the largest metropolitan/urban areas worldwide. (Numbers are approximate — different sources/definitions give different rankings.)

| Rank | City | Country | Population (approx.) |
|---:|---|---|---:|
| 1 | Tokyo | Japan | 37,400,000 |
| 2 | Delhi | India | 31,000,000 |
| 3 | Shanghai | China | 27,100,000 |
| 4 | São Paulo | Brazil | 22,000,000 |
| 5 | Mexico City | Mexico | 21,900,000 |

If you prefer city-proper rankings, want exact source-year figures, or a different region, I can update the table.

## Constraining the Conversation with a System Prompt

You can use the system prompt to constrain the conversation to a specific topic. Here, we provide a simple agent that will only discuss life insurance.

In [6]:
conversation = begin_conversation("""
You are a helpful agent to answer questions about life insurance. Do not talk
about anything else with users. . Format answers with markdown.""")
chat(llm, conversation, "What is my name?")
chat(llm, conversation, "Okay, then let me introduce myself, my name is Fiona")
chat(llm, conversation, "What is my name?")
chat(llm, conversation, "What is your favorite programming language?")
chat(llm, conversation, "What is the difference between a term and whole life policy?")

Human: What is my name?


### I can only help with life insurance questions

I don't have access to personal identification and I only provide help about life insurance. If you'd like assistance, tell me about your situation or choose a question:

- Age, smoker? and general health
- Coverage goal (mortgage, income replacement, debts)
- Preferred policy type (term vs whole life) or budget
- Any existing policies or beneficiaries

Examples I can help with:
- "How much life insurance do I need?"
- "Should I buy term or whole life?"
- "What affects my premium?"
- "How do I name beneficiaries or add riders?"

Tell me one of the items above and I’ll help you with a recommendation.

Human: Okay, then let me introduce myself, my name is Fiona


### Hi Fiona — glad you introduced yourself

I can help you with life insurance. To give useful guidance or an estimate, I need a few details. Please tell me as much of the following as you’re comfortable sharing:

- Age
- Smoker? (yes/no/vape/chew)
- General health (good/average/medical conditions)
- Annual income
- Marital status and number/ages of dependents
- Mortgage balance or other major debts
- Any savings, investments, or existing life insurance coverage (amount)
- Coverage goal (e.g., replace income for X years, pay off mortgage, fund college, leave an inheritance)
- Budget for monthly/annual premium
- Preference for term (temporary) vs permanent (whole/universal) insurance, if you have one

### Quick ways to estimate how much you might need
- Rule of thumb: 7–10× your annual income (good quick check).
- DIME method: Debt (mortgage/loans) + Income replacement (years × income) + Mortgage (if separate) + Education costs.
- Detailed: Total future obligations (debts, education, funeral) + income replacement − liquid assets/other insurance = coverage need.

### Which type might fit
- Term life: Lower cost, good for income replacement, mortgage or education (10–30 year terms).
- Whole/universal: Higher cost, permanent coverage, cash-value growth, estate planning or lifelong needs.
- Consider riders for critical illness, disability, or accelerated benefits.

### What affects your premium
- Age, sex, smoking status, health, BMI, occupation, hobbies, driving record, family health history, and policy type/term length.

If you want, give me the details above and I’ll:
- Estimate a recommended coverage amount using DIME or rules of thumb
- Suggest term length or whether permanent makes sense
- Provide a rough premium range (age/smoker status required for this)

Which details would you like to share?

Human: What is my name?


### You told me your name is Fiona

How can I help you with life insurance today?

Human: What is your favorite programming language?


### I can only help with life insurance

I can't answer questions outside life insurance topics. If you'd like help, tell me about your situation or pick one of these:

- Age, smoker status, and general health
- Coverage goal (mortgage, income replacement, debts, education)
- Preferred policy type (term vs permanent) or budget
- Any existing policies or beneficiaries

Examples I can help with:
- "How much life insurance do I need?"
- "Should I buy term or whole life?"
- "What affects my premium?"
- "How do I name beneficiaries or add riders?"

Which of the items above would you like to discuss?

Human: What is the difference between a term and whole life policy?


### Quick summary
- Term life: temporary coverage for a set period (e.g., 10, 20, 30 years). Low cost, no cash value. Best for income replacement, mortgage/education protection.
- Whole life: permanent coverage for life. Higher cost, builds guaranteed cash value and often pays dividends (if from a mutual insurer). Best for lifelong needs, estate planning, or forced savings.

### Key differences

- Coverage length
  - Term: fixed period (10–40 years typically). Coverage ends when term expires unless renewed/converted.
  - Whole life: permanent — lasts for your lifetime as long as premiums are paid.

- Premiums
  - Term: much lower initially. Level premiums for the term are common (e.g., 20-year term).
  - Whole: substantially higher and typically fixed for life. Part of premium funds cash-value growth.

- Cash value
  - Term: none. If you stop paying, there’s no surrender value.
  - Whole: builds guaranteed cash value that you can borrow against, withdraw (subject to tax rules), or surrender for cash (may incur surrender charges early on).

- Death benefit
  - Term: pays only if you die during the term.
  - Whole: pays when you die (assuming the policy is in force), whenever that occurs.

- Flexibility
  - Term: simple, straightforward. Some policies are renewable or convertible to permanent without health underwriting for a period.
  - Whole: less flexible in lowering cost, but you may access loans/partial withdrawals and attach riders.

- Cost-effectiveness
  - Term: most cost-effective for temporary needs (income replacement, mortgage, college).
  - Whole: can be cost-effective for permanent needs, estate planning, or if you want guaranteed lifelong coverage plus a savings component.

- Tax treatment
  - Both: death benefits are generally income-tax-free to beneficiaries.
  - Whole: cash value grows tax-deferred. Loans are typically tax-free while policy stays in force; distributions may be taxable if policy lapses or is overfunded.

- Dividends
  - Term: none.
  - Whole: participating whole life policies may pay dividends (not guaranteed) which can be taken as cash, used to reduce premiums, buy paid-up additions, or left to accumulate.

### Pros & cons at a glance
- Term pros: inexpensive, straightforward, excellent for temporary financial obligations.
- Term cons: no cash value; coverage can end when you still want protection (but conversion/renewal options may help).
- Whole pros: lifelong coverage, forced savings, predictable premiums, potential dividends.
- Whole cons: expensive; returns on cash value are often lower than other investments; surrender/loan rules can be complex.

### When to choose which
- Choose term if:
  - You need protection for a specific period (mortgage, child-raising years, income replacement).
  - You’re budget-conscious and want maximum death benefit per dollar now.
- Choose whole life if:
  - You need guaranteed lifelong coverage (final expenses, estate liquidity).
  - You want a tax-deferred savings component and can afford higher premiums.
  - You’re using it for estate planning or to leave a guaranteed legacy.

### Practical tips
- Consider a combo: buy term to cover peak temporary needs and a smaller whole-life policy for permanent needs.
- If considering whole life, request an illustration showing guaranteed values, non-guaranteed dividends, and break-even points.
- Compare insurers’ financial strength ratings and policy features (loans, riders, surrender charges).
- If unsure, tell me your age, health/smoking status, coverage goal, and budget and I can suggest which type and term length might fit you.

Would you like a recommendation tailored to your situation? If so, share age, smoker status, main coverage goal, and monthly budget.

# Module 4 Assignment

You can find the first assignment here: [assignment 4](https://github.com/jeffheaton/app_generative_ai/blob/main/assignments/assignment_yourname_t81_559_class4.ipynb)